In [ ]:
!pip install xformers==0.0.28.post3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 103.8 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [6]:
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install --no-deps xformers "trl<0.9.0" peft accelerate bitsandbytes
!pip install --upgrade trl peft transformers accelerate unsloth

  Cloning https://github.com/unslothai/unsloth.git to /tmp/pip-install-wjqw4nd2/unsloth_25abd3aefae4413ba81233456ee785dd
  Running command git clone --filter=blob:none --quiet https://github.com/unslothai/unsloth.git /tmp/pip-install-wjqw4nd2/unsloth_25abd3aefae4413ba81233456ee785dd
  Resolved https://github.com/unslothai/unsloth.git to commit f1f283a24e2dcda1166f45ddafa73e53f8d38c2d
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached trl-1.15.0-py3-none-any.whl.metadata (12 kB)
Using cached trl-1.15.0-py3-none-any.whl (945 kB)
  Attempting uninstall: trl
    Found existing installation: trl 0.8.6
    Uninstalling trl-0.8.6:
      Successfully uninstalled trl-0.8.6
  Using cached trl-0.8.6-py3-none-any.whl.metadata (11 kB)
Using cached trl-0.8.6-py3-none-any.whl (245 kB)
  Attempting uninstall: trl
    Found existing installation: trl 1.15.0
    Uninstalling trl-1.15.0:
      Successfully 

In [10]:
!git clone https://github.com/mykytovych/llm-style-adapter-pipeline.git
%cd llm-style-adapter-pipeline
!mkdir -p data/raw data/processed

Cloning into 'llm-style-adapter-pipeline'...
remote: Enumerating objects: 14, done.
remote: Counting objects: 100% (14/14), done.
remote: Compressing objects: 100% (11/11), done.
remote: Total 14 (delta 1), reused 13 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (14/14), done.
Resolving deltas: 100% (1/1), done.
/content/llm-style-adapter-pipeline/llm-style-adapter-pipeline/llm-style-adapter-pipeline


In [18]:
!git pull origin main

remote: Enumerating objects: 10, done.
remote: Counting objects: 100% (10/10), done.
remote: Compressing objects: 100% (7/7), done.
remote: Total 8 (delta 2), reused 4 (delta 1), pack-reused 0 (from 0)
Unpacking objects: 100% (8/8), 9.60 KiB | 3.20 MiB/s, done.
From https://github.com/mykytovych/llm-style-adapter-pipeline
 * branch            main       -> FETCH_HEAD
   d442d81..ab6c477  main       -> origin/main
Updating d442d81..ab6c477
Fast-forward
 notebooks/01_test_pipeline.ipynb | 528 +++++++++++++++++++++++++++++++++++++++
 scripts/download_hf_dataset.py   |  46 ++++
 2 files changed, 574 insertions(+)
 create mode 100644 notebooks/01_test_pipeline.ipynb
 create mode 100644 scripts/download_hf_dataset.py


In [19]:
!python scripts/download_hf_dataset.py \
  --dataset_name "harpreetsahota/modern-to-shakesperean-translation" \
  --prompt_col "modern" \
  --response_col "shakespearean" \
  --output_csv "data/raw/shakespeare.csv"

README.md: 100% 2.65k/2.65k [00:00<00:00, 6.86MB/s]

data/train-00000-of-00001.parquet: downloading bytes:   0% 0.00/22.8k [00:00<?, ?B/s]
data/train-00000-of-00001.parquet: downloading bytes: 100% 22.8k/22.8k [00:02<00:00, 7.81kB/s, 2.12kB/s  ]
data/train-00000-of-00001.parquet: reconstructing file: 100% 22.8k/22.8k [00:02<00:00, 7.80kB/s, 2.12kB/s  ]
Generating train split: 100% 274/274 [00:00<00:00, 4267.09 examples/s]
Successfully saved 274 samples to data/raw/shakespeare.csv


In [20]:
!python scripts/prepare_dataset.py \
  --input data/raw/shakespeare.csv \
  --output data/processed/shakespeare_train.jsonl \
  --system-prompt "You are William Shakespeare. Answer in Early Modern English with poetic and dramatic flair."

Loading data from data/raw/shakespeare.csv...
Successfully saved 274 samples to data/processed/shakespeare_train.jsonl


In [21]:
!python scripts/train.py \
  --dataset data/processed/shakespeare_train.jsonl \
  --max_steps 60

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: xformers is installed but cannot load with torch 2.11.0+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading model: unsloth/llama-3-8b-Instruct-bnb-4bit
==((====))==  Unsloth 2026.10.7: Fast Llama patching. Transformers: 5.20.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
/usr/local/lib/python3.13/dist-packages/huggingface_hub/c

In [22]:
!python scripts/generate.py \
  --adapter_path "adapter_model" \
  --system_prompt "You are William Shakespeare. Answer in Early Modern English with poetic and dramatic flair." \
  --prompt "What do you think about modern smartphones?"

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: xformers is installed but cannot load with torch 2.11.0+cu130 (it was built for 2.10.0+cu128), so PyTorch attention is used instead.
To install a matching xformers, run `unsloth install-kernels xformers`, or `pip uninstall xformers` to hide this. UNSLOTH_ENABLE_LOGGING=1 shows the full error.
🦥 Unsloth Zoo will now patch everything to make training faster!
Loading model and adapter from: adapter_model...
==((====))==  Unsloth 2026.10.7: Fast Llama patching. Transformers: 5.20.2.
   \\   /|    Tesla T4. Num GPUs = 1. Max memory: 14.563 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu130. CUDA: 7.5. CUDA Toolkit: 13.0. Triton: 3.6.0
\        /    Bfloat16 = FALSE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Loading weights: 100% 291/291 [00:10<00:00, 27.63it/s]
Unslo